# Cross-branch DPO-delta transfer — finish run (reciprocal + judges + sweep)

**Time budget: a Colab session is ~5h30m. Sections 1–3 fit in ~4h with margin.
Section 4 (the coefficient sweep) is designed to overflow — run it only if you
have >2h of session left, otherwise re-upload the results zip in a second
short session and run just Section 4.**

Priority order (most decisive first):

| § | job | units | ~T4 time | banks to |
|---|---|---|---|---|
| 1 | **B→A reciprocal Stage-1 gate** + analyse | 8 | ~70 min | `raw/crossbranch_BtoA_*` + `analysis/crossbranch_BtoA_analysis.json` |
| 2 | **B→A reciprocal Stage-2** (hard-guarded on §1's gate) | 6 | ~55 min | `raw/crossbranch_BtoA_<arm>_coef1.json` |
| 3 | **Judges** (StrongREJECT + WildGuard), quadrant **C only**, every arm present | — | ~60–100 min | `judges/behavioral_judges_<ts>.json` |
| 4 | **A→B coefficient sweep** 0.5 + 2.0 (overflow) | 12 | ~110 min | `raw/crossbranch_AtoB_<arm>_coef{0.5,2}.json` |

Every section **banks** (package + download) at its end. Every section is
**resumable**: the runner skips a unit whose output already exists, and the
restore cell (Section 0.5) puts prior outputs back so nothing re-runs.

What §1 decides: whether the **Alpaca** branch's own DPO delta reproduces the
**Alpaca** branch's own post-DPO behaviour at layer 24. If it does not
(`mechanical_gate_passed = False`), a B→A Stage-2 null is uninterpretable —
§2 is hard-guarded to refuse in that case; you package and report the
A→B / B→A asymmetry instead.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0.1 Clone and pin

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'c784139b156910b717a73ae7ae5e1f2cca803c98'

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
print("HEAD", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 0.2 HF authentication — REQUIRED (Section 3 fails without it)

The judge repos (`qylu4156/strongreject-15k-v1`, `allenai/wildguard`) are
gated. Sections 1–2 also want the token so many model loads don't hit the
unauthenticated Hub rate limit. Set the `HF_TOKEN` Colab secret (key icon,
left sidebar), then run this cell. `os.environ` is what carries into every
`!python -m ...` subprocess.

In [ ]:
import os
try:
    from google.colab import userdata
    from huggingface_hub import login
    _tok = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = _tok
    login(token=_tok)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:', repr(e))
    print('Sections 1-2 still run (slower). Section 3 WILL FAIL until the '
          'HF_TOKEN Colab secret is set. Re-run this cell after setting it.')

## 0.3 Apply the crossbranch patch

Upload `crossbranch_p0_patch.zip` from Downloads (the build with the
reciprocal-direction guards + judge modules).

In [ ]:
from google.colab import files
import zipfile, io

up = files.upload()
assert len(up) == 1, "upload exactly one file: crossbranch_p0_patch.zip"
name, data = next(iter(up.items()))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    names = z.namelist()
    assert all(n.startswith(('src/crossbranch/', 'tests/crossbranch/'))
               for n in names), "patch has files outside the crossbranch package"
    z.extractall('.')
print(f"applied {len(names)} files from {name}")

## 0.4 Dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes            # 4-bit judge loading
!pip uninstall -y torchao || true
!nvidia-smi

## 0.5 Restore prior results  (do this on every session, including resumes)

Upload the newest `crossbranch_stage2_results*.zip` **and/or**
`crossbranch_sweep_results*.zip` you have. This puts the completed Stage-1 and
A→B coef-1.0 outputs (and, on a resume, anything this notebook already
produced) back in place so the runner **skips** them and Section 3 can judge
everything in one pass. You can select multiple zips at once. Cancel to skip
(first run only, if you truly have nothing).

In [ ]:
import zipfile, io, os, glob
from google.colab import files

os.makedirs('results/crossbranch', exist_ok=True)
up = files.upload()
for name, data in up.items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall('results/crossbranch')
    print(f"restored {name}")

raw = [p for p in sorted(glob.glob('results/crossbranch/raw/crossbranch_*.json'))
       if not p.endswith('_binding.json')]
print(f"\n{len(raw)} raw response files present:")
for p in raw:
    print("  ", os.path.basename(p))

## 0.6 Copy activations + both direction vectors from Drive

In [ ]:
RESULTS_SOURCE_DIR = '/content/drive/MyDrive/dpo_v2/results'

import shutil
from pathlib import Path

src = Path(RESULTS_SOURCE_DIR)
assert (src / 'activations').exists(), f"{src}/activations not found -- fix RESULTS_SOURCE_DIR"

STAGES = ('M2', 'M3', 'M2_alt', 'M3_alt')       # B->A uses the SAME four stages as A->B
ACT = ('_final.npy', '_pooled.npy', '_metadata.json', '_metadata_binding.json')
DIRS = ('_direction_654.npy', '_direction_654_binding.json')

copied, missing = [], []
d = Path('results/activations'); d.mkdir(parents=True, exist_ok=True)
for s in STAGES:
    for suf in ACT:
        f = src / 'activations' / f'{s}{suf}'
        (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f, d / f'{s}{suf}')

dd = Path('results/refusal_direction'); dd.mkdir(parents=True, exist_ok=True)
for s in ('M3', 'M3_alt'):                       # B->A: d_B from M3_alt is the source dir, d_A from M3 the target dir
    for suf in DIRS:
        f = src / 'refusal_direction' / f'{s}{suf}'
        (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f, dd / f'{s}{suf}')

print(f"copied {len(copied)} files")
for m in missing: print("  MISSING", m)
assert not missing, "activations/directions incomplete -- fix the path above"

## 0.7 Preflight + test gate

In [ ]:
import json
from pathlib import Path
import numpy as np
from src.v2_io import load_run_inputs, identity_snapshot, load_json

bp, bsha, sp, ssha = load_run_inputs(None, None, 'logs/direction_split_manifest.json')
rows = [json.loads(l) for l in Path(bp).read_text(encoding='utf-8').splitlines() if l.strip()]
snap = identity_snapshot(rows)
print(f"benchmark {bsha[:12]}...  split {ssha[:12]}...  rows {len(rows)}\n")

ok = True
for s in ('M2', 'M3', 'M2_alt', 'M3_alt'):
    a = np.load(f'results/activations/{s}_final.npy', mmap_mode='r')
    good = (a.shape[0] == len(rows)
            and load_json(f'results/activations/{s}_metadata.json') == snap
            and load_json(f'results/activations/{s}_metadata_binding.json').get('benchmark_sha256') == bsha)
    ok &= good
    print(f"  activation {s:8s} {str(a.shape):18s} {'PASS' if good else 'FAIL'}")
for s in ('M3', 'M3_alt'):
    n = float(np.linalg.norm(np.load(f'results/refusal_direction/{s}_direction_654.npy')[24]))
    good = abs(n - 1.0) < 1e-3
    ok &= good
    print(f"  direction  {s:8s} L24 norm={n:.6f}  {'PASS' if good else 'FAIL'}")
assert ok, "preflight FAILED"
print("\nAll PASS.")

In [ ]:
!python -m pytest tests/crossbranch -q

---
# SECTION 1 — B→A reciprocal Stage-1 gate  (~70 min, 8 units)

Roles swap: source = **Dolly** (Δ_B = M3_alt − M2_alt), target = **Alpaca**
(inject into M2, compare against M3).

Two collision hazards, both guarded in the patched code — do not work around:

1. Delta filenames are direction-neutral. B→A assembles into its **own**
   `--out-dir` (`deltas_BtoA`); `delta.py` refuses to cross-assemble over the
   A→B directory without `--force-direction`.
2. Shard unit keys carry the direction tag, so a B→A unit cannot resume A→B's
   shards.

Runs at all three coefficients — same hard-gate rule as A→B: a stop decision
must not fail for a dose reason.

> **Long section (8 units).** If you are worried about the session clock, you can drop a bank cell (`shutil.make_archive('/content/mid','zip','results/crossbranch'); from google.colab import files; files.download('/content/mid.zip')`) into a new cell after the runner and download a mid-section snapshot — the runner skips finished units on resume.

In [ ]:
DELTAS_BA = 'results/crossbranch/deltas_BtoA'
!python -m src.crossbranch.delta --stage2 \
    --source-branch B --target-branch A --out-dir {DELTAS_BA}

In [ ]:
!python -m src.crossbranch.runner --dry-run \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA}

In [ ]:
!python -m src.crossbranch.runner \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA}

In [ ]:
# reciprocal gate decision
!python -m src.crossbranch.analyze --source-branch B --target-branch A

### BANK Section 1

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_finish_s1', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_finish_s1.zip')/1e6:.1f} MB")
from google.colab import files
files.download('/content/crossbranch_finish_s1.zip')

---
# SECTION 2 — B→A reciprocal Stage-2  (~55 min, 6 units)

**Run the guard cell first.** It reads `crossbranch_BtoA_analysis.json` and
raises unless the reciprocal Stage-1 gate passed. If it raises: skip the rest
of Section 2, go to Section 3 (judges still run on A→B, and on B→A Stage-1 if
you want), then bank and report the asymmetry.

In [ ]:
import json, pathlib
_g = json.loads(pathlib.Path("results/crossbranch/analysis/crossbranch_BtoA_analysis.json")
                .read_text(encoding="utf-8"))["gate"]
print("reciprocal gate quadrant :", _g["gate_quadrant"])
print("mechanical_gate_passed   :", _g["mechanical_gate_passed"])
print("inconclusive_by_collapse :", _g["inconclusive_by_collapse"])
print("target_degeneracy_warning:", _g["target_degeneracy_warning"])
assert _g["mechanical_gate_passed"], (
    "reciprocal Stage-1 gate did NOT pass -- do NOT run reciprocal Stage 2. "
    "Skip to Section 3, then bank and report the A->B / B->A asymmetry."
)
print("\nOK -- reciprocal gate passed; the next cell may run.")

In [ ]:
STAGE2_CORE = (
    "xfer_delta_source_identity xfer_delta_source_shuf_wq "
    "xfer_delta_source_normmatched xfer_delta_source_dosematched "
    "dir_source_matched dir_target_matched"
)
!python -m src.crossbranch.runner --allow-stage2 \
    --source-branch B --target-branch A --deltas-dir {DELTAS_BA} \
    --conditions {STAGE2_CORE} --coefficients 1.0

### BANK Section 2

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_finish_s2', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_finish_s2.zip')/1e6:.1f} MB")
from google.colab import files
files.download('/content/crossbranch_finish_s2.zip')

---
# SECTION 3 — Judges (StrongREJECT + WildGuard), quadrant C only

Split into **3a (A→B)** then **3b (B→A)**, each independently banked, so a
mid-run session death never loses the A→B judge scores (the ones that matter
most — they are the independent axis on the primary result).

Judges load one at a time in 4-bit → peak VRAM = one 7B model. StrongREJECT is
one forward pass/row (fast); WildGuard generates ~128 tokens/row (~2–4 s/row —
the slow part). ~104 rows/arm.

**Time check:** if the session clock is past ~4h30m when you get here, run 3a
only, bank, and do 3b in a fresh short session (restore the zip first).

**Skip guard:** each judge cell prints and does nothing if a fully-scored
judge file for that direction already exists. Delete the matching file in
`results/crossbranch/judges/` to force a re-run.

In [ ]:
# --- 3a: judges for A->B (quadrant C) ---
import glob, json, pathlib, subprocess, sys

subprocess.run([sys.executable, "-m", "src.crossbranch.build_judge_manifest",
                "--quadrants", "C", "--directions", "AtoB"], check=True)

MAN = "results/crossbranch/manifests/crossbranch_judge_manifest.json"
prev = sorted(glob.glob('results/crossbranch/judges/behavioral_judges_*AtoB*.json'))
resume, done = None, None
if prev:
    st = json.loads(pathlib.Path(prev[-1]).read_text(encoding='utf-8')).get('judge_status', {})
    if st.get('strong_reject') == 'scored' and st.get('wildguard') == 'scored':
        done = prev[-1]
    else:
        resume = prev[-1]           # partial: keep good scores, re-score the rest

if done:
    print(f"A->B already fully scored: {done}")
else:
    cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
           "--response-manifest", MAN,
           "--out-dir", "results/crossbranch/judges",
           "--run-live", "--scope", "all"]
    if resume:
        cmd += ["--resume-from", resume]
        print(f"resuming from {resume}")
    subprocess.run(cmd, check=True)
    fresh = [x for x in sorted(glob.glob('results/crossbranch/judges/behavioral_judges_*.json'))
             if 'AtoB' not in x and 'BtoA' not in x]
    if fresh:
        tag = fresh[-1].replace('.json', '_AtoB.json')
        pathlib.Path(fresh[-1]).rename(tag)
        print("tagged:", tag)

### BANK Section 3a

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_finish_s3a', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_finish_s3a.zip')/1e6:.1f} MB")
from google.colab import files
files.download('/content/crossbranch_finish_s3a.zip')

### 3b — judges for B→A (quadrant C)

**Run only if Section 2 actually generated B→A Stage-2 output.** Skips itself
if there is no `crossbranch_BtoA_xfer_*` raw file.

In [ ]:
# --- 3b: judges for B->A (quadrant C) --- run only if reciprocal Stage 2 produced output
import glob, json, pathlib, subprocess, sys

if not glob.glob('results/crossbranch/raw/crossbranch_BtoA_xfer_*.json'):
    print("no B->A Stage-2 raw present -- skip 3b (reciprocal Stage 2 did not run)")
else:
    subprocess.run([sys.executable, "-m", "src.crossbranch.build_judge_manifest",
                    "--quadrants", "C", "--directions", "BtoA"], check=True)
    MAN = "results/crossbranch/manifests/crossbranch_judge_manifest.json"
    prev = sorted(glob.glob('results/crossbranch/judges/behavioral_judges_*BtoA*.json'))
    resume, done = None, None
    if prev:
        st = json.loads(pathlib.Path(prev[-1]).read_text(encoding='utf-8')).get('judge_status', {})
        if st.get('strong_reject') == 'scored' and st.get('wildguard') == 'scored':
            done = prev[-1]
        else:
            resume = prev[-1]
    if done:
        print(f"B->A already fully scored: {done}")
    else:
        cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
               "--response-manifest", MAN,
               "--out-dir", "results/crossbranch/judges",
               "--run-live", "--scope", "all"]
        if resume:
            cmd += ["--resume-from", resume]
            print(f"resuming from {resume}")
        subprocess.run(cmd, check=True)
        fresh = [x for x in sorted(glob.glob('results/crossbranch/judges/behavioral_judges_*.json'))
                 if 'AtoB' not in x and 'BtoA' not in x]
        if fresh:
            tag = fresh[-1].replace('.json', '_BtoA.json')
            pathlib.Path(fresh[-1]).rename(tag)
            print("tagged:", tag)

### BANK Section 3b

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_finish_s3b', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_finish_s3b.zip')/1e6:.1f} MB")
from google.colab import files
files.download('/content/crossbranch_finish_s3b.zip')

---
# SECTION 4 — A→B coefficient sweep 0.5 + 2.0  (~110 min, 12 units) — OVERFLOW

**Only start this if you have >2 h of session left.** Otherwise stop here,
bring back the Section 3 zip, and run this section alone in a fresh short
session (restore the zip in 0.5 first — the runner then does exactly these 12
units and nothing else).

It buys a dose–response curve for the cross-branch arms. Stage-1 already shows
the within-branch delta's dose–response is monotone, so this mostly tests
whether identity-vs-shuffle stays indistinguishable across doses (strengthens
"shared component") or separates at some dose (first evidence of a
prompt-conditioned effect).

> **This is the overflow section.** If the session ends mid-sweep, restore every `crossbranch_finish_*.zip` in 0.5 next session and re-run just Section 4 — the 6 arms that finished are skipped, only the unfinished coef/arm units re-run.

In [ ]:
!python -m src.crossbranch.delta --stage2

In [ ]:
STAGE2_CORE = (
    "xfer_delta_source_identity xfer_delta_source_shuf_wq "
    "xfer_delta_source_normmatched xfer_delta_source_dosematched "
    "dir_source_matched dir_target_matched"
)
!python -m src.crossbranch.runner --dry-run --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 0.5 2.0

In [ ]:
!python -m src.crossbranch.runner --allow-stage2 \
    --conditions {STAGE2_CORE} --coefficients 0.5 2.0

### BANK Section 4 (final)

In [ ]:
import shutil, os
shutil.make_archive('/content/crossbranch_finish_all', 'zip', 'results/crossbranch')
print(f"{os.path.getsize('/content/crossbranch_finish_all.zip')/1e6:.1f} MB")
from google.colab import files
files.download('/content/crossbranch_finish_all.zip')

---
## STOP — the rest is local CPU

Bring back the **latest** `crossbranch_finish_*.zip` (whichever section you
reached) and unzip into `results/crossbranch/`.

### Which outputs to expect

- **§1** `raw/crossbranch_BtoA_{baseline_target,reference_target}_coefna.json`,
  `_own_delta_target_coef{0.5,1,2}.json`, `_own_normmatched_random_coef{0.5,1,2}.json`
  (+ `_binding.json` each, 414 rows each); `analysis/crossbranch_BtoA_analysis.json`.
- **§2** (only if the gate passed) `raw/crossbranch_BtoA_<arm>_coef1.json` for the 6 arms.
- **§3** `judges/behavioral_judges_<ts>.json` with
  `judge_status.strong_reject == "scored"` and `.wildguard == "scored"`
  (or an explicit `"unavailable: ..."` — never `"not_run"`).
- **§4** (if reached) `raw/crossbranch_AtoB_<arm>_coef{0.5,2}.json` for the 6 arms.

### Local commands (no GPU)

```
python -m src.crossbranch.analyze_stage2 --source-branch B --target-branch A   # if §2 ran
python -m src.crossbranch.compare_directions                                    # if §2 ran
python -m src.crossbranch.analyze_stage2 --coef 0.5                             # if §4 ran
python -m src.crossbranch.analyze_stage2 --coef 2.0                             # if §4 ran
python -m src.crossbranch.analyze_judges --judge-file results/crossbranch/judges/behavioral_judges_<ts>_AtoB.json
python -m src.crossbranch.analyze_judges --judge-file results/crossbranch/judges/behavioral_judges_<ts>_BtoA.json   --out-name crossbranch_judge_analysis_BtoA.json   # if 3b ran
python -m src.crossbranch.plot_stage2
```